In [ ]:

"""
CN7 04 - Validation 설계 + 첫 Baseline
=======================================

목표
----
1) CN7_03에서 복원한 raw 1211행 사용
2) 완전히 동일한 sensor vector를 하나의 Group으로 묶음
3) 같은 Group이 train/valid에 갈라지지 않도록 StratifiedGroupKFold
4) 시간구간을 분리한 Purged Time-Block CV 추가
5) Sensors only vs Sensors + Side 비교
6) Logistic Regression / ExtraTrees 비교
7) PR-AUC 중심 평가
8) strict forward validation이 가능한지도 별도로 진단

중요
----
- Reason / TimeStamp / 날짜 / Serial은 예측변수로 넣지 않는다.
- PassOrFail=1을 불량으로 사용한다.
- 동일 센서벡터는 반드시 같은 fold에 둔다.
- Time-block CV는 "양방향 blocked robustness"이며,
  strict 미래예측 검증과는 구분한다.
"""

from pathlib import Path
import warnings
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
)

warnings.filterwarnings("ignore")

# =============================================================================
# 0. 경로 / 설정
# =============================================================================

CN7_WORK_DIR = Path(
    r"../../outputs/research"
)

RAW_PATH = (
    CN7_WORK_DIR
    / "cn7_03_eda"
    / "cn7_train_raw_1211.csv"
)

OUT_DIR = CN7_WORK_DIR / "cn7_04_validation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "PassOrFail"

RANDOM_SEEDS = [
    42, 52, 62, 72, 82,
    92, 102, 112, 122, 132,
]

N_SPLITS = 5

# 시간 블록
TIME_BLOCK_MINUTES = 10
PURGE_MINUTES = 10

# 모델
N_TREES = 400

# OOF bootstrap CI
BOOTSTRAP_N = 500
BOOTSTRAP_SEED = 2026

SENSOR_COLS = [
    "Injection_Time",
    "Filling_Time",
    "Plasticizing_Time",
    "Cycle_Time",
    "Clamp_Close_Time",
    "Cushion_Position",
    "Plasticizing_Position",
    "Clamp_Open_Position",
    "Max_Injection_Speed",
    "Max_Screw_RPM",
    "Average_Screw_RPM",
    "Max_Injection_Pressure",
    "Max_Switch_Over_Pressure",
    "Max_Back_Pressure",
    "Average_Back_Pressure",
    "Barrel_Temperature_1",
    "Barrel_Temperature_2",
    "Barrel_Temperature_3",
    "Barrel_Temperature_4",
    "Barrel_Temperature_5",
    "Barrel_Temperature_6",
    "Hopper_Temperature",
    "Mold_Temperature_3",
    "Mold_Temperature_4",
]


# =============================================================================
# 유틸
# =============================================================================

def banner(title):
    print("\n" + "=" * 110)
    print(title)
    print("=" * 110)


def top_fraction_metrics(y_true, score, frac):
    y_true = np.asarray(y_true).astype(int)
    score = np.asarray(score).astype(float)

    n = len(y_true)
    k = max(1, int(np.ceil(n * frac)))

    order = np.argsort(-score)
    top = order[:k]

    tp = int(y_true[top].sum())
    total_pos = int(y_true.sum())

    precision = tp / k
    recall = tp / total_pos if total_pos > 0 else np.nan

    return {
        f"precision@{int(frac*100)}%": precision,
        f"recall@{int(frac*100)}%": recall,
        f"tp@{int(frac*100)}%": tp,
        f"k@{int(frac*100)}%": k,
    }


def score_predictions(y_true, score):
    y_true = np.asarray(y_true).astype(int)
    score = np.asarray(score).astype(float)

    valid = np.isfinite(score)
    y = y_true[valid]
    s = score[valid]

    out = {
        "n": int(len(y)),
        "defects": int(y.sum()),
        "defect_rate": float(y.mean()) if len(y) else np.nan,
    }

    if len(y) == 0 or len(np.unique(y)) < 2:
        out.update({
            "PR_AUC": np.nan,
            "ROC_AUC": np.nan,
        })
        return out

    out["PR_AUC"] = float(
        average_precision_score(y, s)
    )
    out["ROC_AUC"] = float(
        roc_auc_score(y, s)
    )

    for frac in [0.02, 0.05, 0.10]:
        out.update(
            top_fraction_metrics(y, s, frac)
        )

    return out


def make_model(name, seed):
    if name == "logreg":
        return Pipeline([
            ("scaler", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    C=0.1,
                    class_weight="balanced",
                    max_iter=5000,
                    solver="liblinear",
                    random_state=seed,
                ),
            ),
        ])

    if name == "extratrees":
        return ExtraTreesClassifier(
            n_estimators=N_TREES,
            max_depth=None,
            min_samples_leaf=2,
            max_features="sqrt",
            class_weight="balanced",
            random_state=seed,
            n_jobs=-1,
        )

    raise ValueError(name)


def bootstrap_group_ci(
    y,
    score,
    groups,
    n_boot=500,
    seed=2026,
):
    """
    동일 센서벡터 Group을 sampling unit으로 사용한 bootstrap.
    PR-AUC 불확실성을 대략적으로 확인하기 위한 용도.
    """
    y = np.asarray(y).astype(int)
    score = np.asarray(score).astype(float)
    groups = np.asarray(groups)

    valid = np.isfinite(score)
    y = y[valid]
    score = score[valid]
    groups = groups[valid]

    unique_groups = np.unique(groups)
    group_to_idx = {
        g: np.where(groups == g)[0]
        for g in unique_groups
    }

    rng = np.random.default_rng(seed)

    pr_values = []
    roc_values = []

    for _ in range(n_boot):
        sampled_groups = rng.choice(
            unique_groups,
            size=len(unique_groups),
            replace=True,
        )

        idx_parts = [
            group_to_idx[g]
            for g in sampled_groups
        ]
        idx = np.concatenate(idx_parts)

        yy = y[idx]
        ss = score[idx]

        if len(np.unique(yy)) < 2:
            continue

        pr_values.append(
            average_precision_score(yy, ss)
        )
        roc_values.append(
            roc_auc_score(yy, ss)
        )

    if len(pr_values) == 0:
        return {
            "PR_CI_low": np.nan,
            "PR_CI_high": np.nan,
            "ROC_CI_low": np.nan,
            "ROC_CI_high": np.nan,
            "bootstrap_valid": 0,
        }

    return {
        "PR_CI_low": float(np.quantile(pr_values, 0.025)),
        "PR_CI_high": float(np.quantile(pr_values, 0.975)),
        "ROC_CI_low": float(np.quantile(roc_values, 0.025)),
        "ROC_CI_high": float(np.quantile(roc_values, 0.975)),
        "bootstrap_valid": len(pr_values),
    }


def predict_fold(
    X,
    y,
    train_idx,
    valid_idx,
    model_name,
    seed,
):
    model = make_model(model_name, seed)

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]

    model.fit(X_train, y_train)

    pred = model.predict_proba(X_valid)[:, 1]

    return pred


# =============================================================================
# 1. 로드
# =============================================================================

banner("CN7 04 - Validation 설계 + Baseline")

if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"CN7_03 raw 파일을 찾지 못했습니다:\n{RAW_PATH}"
    )

df = pd.read_csv(RAW_PATH)

print("RAW_PATH:", RAW_PATH)
print("shape   :", df.shape)

if TARGET not in df.columns:
    raise KeyError(f"{TARGET} 없음")

df[TARGET] = pd.to_numeric(
    df[TARGET],
    errors="raise",
).astype(int)

print(
    f"target=1: {df[TARGET].sum()} / {len(df)} "
    f"= {df[TARGET].mean():.3%}"
)

# =============================================================================
# 2. 시간 / Side 정리
# =============================================================================

banner("1) Metadata 정리")

if "TimeStamp_dt" in df.columns:
    df["TimeStamp_dt"] = pd.to_datetime(
        df["TimeStamp_dt"],
        errors="coerce",
    )
elif "TimeStamp" in df.columns:
    df["TimeStamp_dt"] = pd.to_datetime(
        df["TimeStamp"],
        errors="coerce",
    )
else:
    raise KeyError("TimeStamp / TimeStamp_dt 없음")

if df["TimeStamp_dt"].isna().any():
    raise RuntimeError("TimeStamp 파싱 실패 행이 있습니다.")

if "Side" not in df.columns:
    if "PART_NAME" not in df.columns:
        raise KeyError("Side/PART_NAME 없음")

    s = df["PART_NAME"].astype(str).str.upper()
    df["Side"] = np.where(
        s.str.contains(r"\bRH\b", regex=True, na=False),
        "RH",
        np.where(
            s.str.contains(r"\bLH\b", regex=True, na=False),
            "LH",
            "UNKNOWN",
        ),
    )

print("Time:", df["TimeStamp_dt"].min(), "~", df["TimeStamp_dt"].max())
print("\nSide:")
print(df["Side"].value_counts(dropna=False).to_string())

print("\nSide x Target:")
print(
    pd.crosstab(
        df["Side"],
        df[TARGET],
        margins=True,
    ).to_string()
)

# =============================================================================
# 3. 센서 확정 + 상수 제거
# =============================================================================

banner("2) Feature set")

missing_sensors = [
    c for c in SENSOR_COLS
    if c not in df.columns
]

if missing_sensors:
    raise KeyError(
        f"다음 sensor 컬럼이 없습니다: {missing_sensors}"
    )

constant_cols = [
    c for c in SENSOR_COLS
    if df[c].nunique(dropna=False) <= 1
]

sensor_cols = [
    c for c in SENSOR_COLS
    if c not in constant_cols
]

print("원래 sensors :", len(SENSOR_COLS))
print("상수 제거     :", constant_cols)
print("사용 sensors :", len(sensor_cols))

# Side binary
df["Side_RH"] = (df["Side"] == "RH").astype(int)

FEATURE_SETS = {
    "sensors_only": sensor_cols,
    "sensors_plus_side": sensor_cols + ["Side_RH"],
}

for name, cols in FEATURE_SETS.items():
    print(f"{name:20s}: {len(cols)} features")

# =============================================================================
# 4. 동일 sensor vector Group 생성
# =============================================================================

banner("3) 동일 sensor vector Group")

# exact raw sensor vector 기준
df["sensor_group_id"] = (
    df.groupby(
        sensor_cols,
        dropna=False,
        sort=False,
    )
    .ngroup()
)

group_stats = (
    df.groupby("sensor_group_id")
    .agg(
        rows=(TARGET, "size"),
        defects=(TARGET, "sum"),
        target_nunique=(TARGET, "nunique"),
        side_nunique=("Side", "nunique"),
        first_time=("TimeStamp_dt", "min"),
        last_time=("TimeStamp_dt", "max"),
    )
    .reset_index()
)

print("전체 group:", group_stats["sensor_group_id"].nunique())
print(
    "2행 이상 group:",
    int((group_stats["rows"] >= 2).sum())
)
print(
    "positive 포함 group:",
    int((group_stats["defects"] >= 1).sum())
)
print(
    "same-X/different-target group:",
    int((group_stats["target_nunique"] > 1).sum())
)

group_stats.to_csv(
    OUT_DIR / "sensor_group_stats.csv",
    index=False,
    encoding="utf-8-sig",
)

# 같은 group이 시간상 얼마나 떨어져있는지
group_stats["duration_seconds"] = (
    group_stats["last_time"]
    - group_stats["first_time"]
).dt.total_seconds()

print("\nGroup time-span:")
print(
    group_stats["duration_seconds"]
    .describe()
    .to_string()
)

# =============================================================================
# 5. Repeated StratifiedGroupKFold
# =============================================================================

banner("4) Repeated StratifiedGroupKFold")

y = df[TARGET]
groups = df["sensor_group_id"]

group_oof_records = []
group_fold_records = []

for feature_name, cols in FEATURE_SETS.items():
    X = df[cols].astype(float)

    for model_name in ["logreg", "extratrees"]:

        pred_sum = np.zeros(len(df), dtype=float)
        pred_count = np.zeros(len(df), dtype=int)

        fold_id_global = 0

        for repeat_no, seed in enumerate(
            RANDOM_SEEDS,
            start=1,
        ):
            cv = StratifiedGroupKFold(
                n_splits=N_SPLITS,
                shuffle=True,
                random_state=seed,
            )

            for fold_no, (tr_idx, va_idx) in enumerate(
                cv.split(X, y, groups),
                start=1,
            ):
                fold_id_global += 1

                tr_groups = set(
                    groups.iloc[tr_idx].tolist()
                )
                va_groups = set(
                    groups.iloc[va_idx].tolist()
                )

                overlap = len(
                    tr_groups.intersection(va_groups)
                )

                if overlap != 0:
                    raise RuntimeError(
                        "Group leakage 발생"
                    )

                y_tr = y.iloc[tr_idx]
                y_va = y.iloc[va_idx]

                if y_tr.nunique() < 2:
                    print(
                        f"[SKIP] {feature_name}/{model_name} "
                        f"repeat={repeat_no} fold={fold_no}: "
                        f"train one-class"
                    )
                    continue

                pred = predict_fold(
                    X=X,
                    y=y,
                    train_idx=tr_idx,
                    valid_idx=va_idx,
                    model_name=model_name,
                    seed=seed + fold_no,
                )

                pred_sum[va_idx] += pred
                pred_count[va_idx] += 1

                fold_metric = score_predictions(
                    y_va.to_numpy(),
                    pred,
                )

                fold_metric.update({
                    "cv": "repeated_stratified_group",
                    "feature_set": feature_name,
                    "model": model_name,
                    "repeat": repeat_no,
                    "fold": fold_no,
                    "train_n": len(tr_idx),
                    "train_pos": int(y_tr.sum()),
                    "valid_n": len(va_idx),
                    "valid_pos": int(y_va.sum()),
                    "train_groups": len(tr_groups),
                    "valid_groups": len(va_groups),
                    "group_overlap": overlap,
                })

                group_fold_records.append(
                    fold_metric
                )

        oof = np.divide(
            pred_sum,
            pred_count,
            out=np.full(
                len(df),
                np.nan,
                dtype=float,
            ),
            where=pred_count > 0,
        )

        overall = score_predictions(
            y.to_numpy(),
            oof,
        )

        ci = bootstrap_group_ci(
            y=y.to_numpy(),
            score=oof,
            groups=groups.to_numpy(),
            n_boot=BOOTSTRAP_N,
            seed=BOOTSTRAP_SEED,
        )

        overall.update(ci)

        print(
            f"{feature_name:20s} | "
            f"{model_name:10s} | "
            f"OOF PR={overall['PR_AUC']:.4f} "
            f"[{overall['PR_CI_low']:.4f}, {overall['PR_CI_high']:.4f}] | "
            f"ROC={overall['ROC_AUC']:.4f}"
        )

        for i in range(len(df)):
            group_oof_records.append({
                "row_index": i,
                "cv": "repeated_stratified_group",
                "feature_set": feature_name,
                "model": model_name,
                "target": int(y.iloc[i]),
                "sensor_group_id": int(groups.iloc[i]),
                "Side": df["Side"].iloc[i],
                "TimeStamp_dt": df["TimeStamp_dt"].iloc[i],
                "prediction": oof[i],
                "prediction_count": int(pred_count[i]),
            })

group_fold_df = pd.DataFrame(
    group_fold_records
)

group_oof_df = pd.DataFrame(
    group_oof_records
)

group_fold_df.to_csv(
    OUT_DIR / "repeated_group_fold_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

group_oof_df.to_csv(
    OUT_DIR / "repeated_group_oof.csv",
    index=False,
    encoding="utf-8-sig",
)

# =============================================================================
# 6. 시간 block 생성
# =============================================================================

banner("5) Time Block 구조")

df["time_block"] = df["TimeStamp_dt"].dt.floor(
    f"{TIME_BLOCK_MINUTES}min"
)

time_block_stats = (
    df.groupby("time_block")
    .agg(
        rows=(TARGET, "size"),
        defects=(TARGET, "sum"),
        groups=("sensor_group_id", "nunique"),
        start=("TimeStamp_dt", "min"),
        end=("TimeStamp_dt", "max"),
    )
    .reset_index()
)

print(
    time_block_stats[
        time_block_stats["defects"] > 0
    ].to_string(index=False)
)

time_block_stats.to_csv(
    OUT_DIR / "time_block_stats.csv",
    index=False,
    encoding="utf-8-sig",
)

# =============================================================================
# 7. Purged Time-Block CV
# =============================================================================

banner("6) Purged Time-Block CV")

block_oof_records = []
block_fold_records = []

unique_blocks = (
    time_block_stats["time_block"]
    .sort_values()
    .tolist()
)

for feature_name, cols in FEATURE_SETS.items():
    X = df[cols].astype(float)

    for model_name in ["logreg", "extratrees"]:

        oof = np.full(
            len(df),
            np.nan,
            dtype=float,
        )

        pred_count = np.zeros(
            len(df),
            dtype=int,
        )

        valid_fold_count = 0

        for fold_no, block in enumerate(
            unique_blocks,
            start=1,
        ):
            block = pd.Timestamp(block)

            block_start = block
            block_end = (
                block
                + pd.Timedelta(
                    minutes=TIME_BLOCK_MINUTES
                )
            )

            purge_start = (
                block_start
                - pd.Timedelta(
                    minutes=PURGE_MINUTES
                )
            )
            purge_end = (
                block_end
                + pd.Timedelta(
                    minutes=PURGE_MINUTES
                )
            )

            val_mask = (
                df["time_block"] == block
            )

            # validation block + 주변 purge 구간 제거
            train_mask = ~(
                (df["TimeStamp_dt"] >= purge_start)
                &
                (df["TimeStamp_dt"] < purge_end)
            )

            val_idx = np.where(
                val_mask.to_numpy()
            )[0]

            # 같은 sensor group이 다른 시간에 존재해도
            # train에서 제거
            val_groups = set(
                df.loc[
                    val_mask,
                    "sensor_group_id",
                ].tolist()
            )

            train_mask = (
                train_mask
                &
                ~df[
                    "sensor_group_id"
                ].isin(val_groups)
            )

            tr_idx = np.where(
                train_mask.to_numpy()
            )[0]

            if len(val_idx) == 0:
                continue

            y_tr = y.iloc[tr_idx]
            y_va = y.iloc[val_idx]

            if y_tr.nunique() < 2:
                block_fold_records.append({
                    "cv": "purged_time_block",
                    "feature_set": feature_name,
                    "model": model_name,
                    "fold": fold_no,
                    "time_block": block,
                    "train_n": len(tr_idx),
                    "train_pos": int(y_tr.sum()),
                    "valid_n": len(val_idx),
                    "valid_pos": int(y_va.sum()),
                    "valid_metric_possible": (
                        y_va.nunique() == 2
                    ),
                    "fit_possible": False,
                    "PR_AUC": np.nan,
                    "ROC_AUC": np.nan,
                })
                continue

            pred = predict_fold(
                X=X,
                y=y,
                train_idx=tr_idx,
                valid_idx=val_idx,
                model_name=model_name,
                seed=42 + fold_no,
            )

            oof[val_idx] = pred
            pred_count[val_idx] += 1
            valid_fold_count += 1

            fold_metric = score_predictions(
                y_va.to_numpy(),
                pred,
            )

            fold_metric.update({
                "cv": "purged_time_block",
                "feature_set": feature_name,
                "model": model_name,
                "fold": fold_no,
                "time_block": block,
                "train_n": len(tr_idx),
                "train_pos": int(y_tr.sum()),
                "valid_n": len(val_idx),
                "valid_pos": int(y_va.sum()),
                "valid_metric_possible": (
                    y_va.nunique() == 2
                ),
                "fit_possible": True,
            })

            block_fold_records.append(
                fold_metric
            )

        overall = score_predictions(
            y.to_numpy(),
            oof,
        )

        ci = bootstrap_group_ci(
            y=y.to_numpy(),
            score=oof,
            groups=groups.to_numpy(),
            n_boot=BOOTSTRAP_N,
            seed=BOOTSTRAP_SEED + 1,
        )

        overall.update(ci)

        print(
            f"{feature_name:20s} | "
            f"{model_name:10s} | "
            f"OOF n={overall['n']:4d} | "
            f"pos={overall['defects']:2d} | "
            f"PR={overall['PR_AUC']:.4f} "
            f"[{overall['PR_CI_low']:.4f}, {overall['PR_CI_high']:.4f}] | "
            f"ROC={overall['ROC_AUC']:.4f}"
        )

        for i in range(len(df)):
            block_oof_records.append({
                "row_index": i,
                "cv": "purged_time_block",
                "feature_set": feature_name,
                "model": model_name,
                "target": int(y.iloc[i]),
                "sensor_group_id": int(groups.iloc[i]),
                "Side": df["Side"].iloc[i],
                "TimeStamp_dt": df["TimeStamp_dt"].iloc[i],
                "time_block": df["time_block"].iloc[i],
                "prediction": oof[i],
                "prediction_count": int(pred_count[i]),
            })

block_fold_df = pd.DataFrame(
    block_fold_records
)

block_oof_df = pd.DataFrame(
    block_oof_records
)

block_fold_df.to_csv(
    OUT_DIR / "purged_block_fold_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

block_oof_df.to_csv(
    OUT_DIR / "purged_block_oof.csv",
    index=False,
    encoding="utf-8-sig",
)

# =============================================================================
# 8. Strict Forward 가능성 진단
# =============================================================================

banner("7) Strict Forward Validation 가능성")

date_col = df["TimeStamp_dt"].dt.date

date_stats = (
    df.assign(Date=date_col)
    .groupby("Date")
    .agg(
        rows=(TARGET, "size"),
        defects=(TARGET, "sum"),
        start=("TimeStamp_dt", "min"),
        end=("TimeStamp_dt", "max"),
    )
    .reset_index()
)

print(date_stats.to_string(index=False))

date_stats.to_csv(
    OUT_DIR / "date_target_stats.csv",
    index=False,
    encoding="utf-8-sig",
)

forward_records = []

dates = sorted(
    pd.Series(date_col).unique()
)

for i in range(1, len(dates)):
    train_dates = dates[:i]
    valid_date = dates[i]

    tr = df[
        date_col.isin(train_dates)
    ]
    va = df[
        date_col == valid_date
    ]

    rec = {
        "train_dates": "|".join(
            map(str, train_dates)
        ),
        "valid_date": str(valid_date),
        "train_n": len(tr),
        "train_pos": int(tr[TARGET].sum()),
        "valid_n": len(va),
        "valid_pos": int(va[TARGET].sum()),
        "fit_possible": tr[TARGET].nunique() == 2,
        "metric_possible": va[TARGET].nunique() == 2,
    }

    rec["strict_forward_valid"] = (
        rec["fit_possible"]
        and rec["metric_possible"]
    )

    forward_records.append(rec)

forward_df = pd.DataFrame(
    forward_records
)

print("\nStrict forward folds:")
print(forward_df.to_string(index=False))

forward_df.to_csv(
    OUT_DIR / "strict_forward_diagnostic.csv",
    index=False,
    encoding="utf-8-sig",
)

# =============================================================================
# 9. 최종 요약표
# =============================================================================

banner("8) 최종 Validation 요약")

summary_records = []

# Group CV
for (
    feature_set,
    model_name
), part in group_oof_df.groupby(
    ["feature_set", "model"]
):
    part = part.sort_values(
        "row_index"
    )

    score = part["prediction"].to_numpy()
    yy = part["target"].to_numpy()
    gg = part["sensor_group_id"].to_numpy()

    metric = score_predictions(
        yy,
        score,
    )

    ci = bootstrap_group_ci(
        yy,
        score,
        gg,
        n_boot=BOOTSTRAP_N,
        seed=BOOTSTRAP_SEED,
    )

    metric.update(ci)

    fold_part = group_fold_df[
        (group_fold_df["feature_set"] == feature_set)
        &
        (group_fold_df["model"] == model_name)
    ]

    valid_fold_pr = fold_part[
        "PR_AUC"
    ].dropna()

    metric.update({
        "cv": "repeated_stratified_group",
        "feature_set": feature_set,
        "model": model_name,
        "fold_PR_mean": (
            valid_fold_pr.mean()
            if len(valid_fold_pr)
            else np.nan
        ),
        "fold_PR_std": (
            valid_fold_pr.std()
            if len(valid_fold_pr) > 1
            else np.nan
        ),
        "n_folds": len(fold_part),
    })

    summary_records.append(metric)

# Block CV
for (
    feature_set,
    model_name
), part in block_oof_df.groupby(
    ["feature_set", "model"]
):
    part = part.sort_values(
        "row_index"
    )

    score = part["prediction"].to_numpy()
    yy = part["target"].to_numpy()
    gg = part["sensor_group_id"].to_numpy()

    metric = score_predictions(
        yy,
        score,
    )

    ci = bootstrap_group_ci(
        yy,
        score,
        gg,
        n_boot=BOOTSTRAP_N,
        seed=BOOTSTRAP_SEED + 1,
    )

    metric.update(ci)

    fold_part = block_fold_df[
        (block_fold_df["feature_set"] == feature_set)
        &
        (block_fold_df["model"] == model_name)
    ]

    valid_fold_pr = fold_part[
        "PR_AUC"
    ].dropna()

    metric.update({
        "cv": "purged_time_block",
        "feature_set": feature_set,
        "model": model_name,
        "fold_PR_mean": (
            valid_fold_pr.mean()
            if len(valid_fold_pr)
            else np.nan
        ),
        "fold_PR_std": (
            valid_fold_pr.std()
            if len(valid_fold_pr) > 1
            else np.nan
        ),
        "n_folds": len(fold_part),
    })

    summary_records.append(metric)

summary_df = pd.DataFrame(
    summary_records
)

order_cols = [
    "cv",
    "feature_set",
    "model",
    "n",
    "defects",
    "defect_rate",
    "PR_AUC",
    "PR_CI_low",
    "PR_CI_high",
    "ROC_AUC",
    "ROC_CI_low",
    "ROC_CI_high",
    "recall@2%",
    "precision@2%",
    "recall@5%",
    "precision@5%",
    "recall@10%",
    "precision@10%",
    "fold_PR_mean",
    "fold_PR_std",
    "n_folds",
]

order_cols = [
    c for c in order_cols
    if c in summary_df.columns
]

summary_df = summary_df[
    order_cols
].sort_values(
    ["cv", "PR_AUC"],
    ascending=[True, False],
)

print(
    summary_df
    .round(4)
    .to_string(index=False)
)

summary_df.to_csv(
    OUT_DIR / "validation_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

# =============================================================================
# 10. Side 추가 효과
# =============================================================================

banner("9) Side 추가 효과")

side_effect_records = []

for cv_name in summary_df["cv"].unique():
    for model_name in summary_df["model"].unique():
        a = summary_df[
            (summary_df["cv"] == cv_name)
            &
            (summary_df["model"] == model_name)
            &
            (summary_df["feature_set"] == "sensors_only")
        ]

        b = summary_df[
            (summary_df["cv"] == cv_name)
            &
            (summary_df["model"] == model_name)
            &
            (summary_df["feature_set"] == "sensors_plus_side")
        ]

        if len(a) != 1 or len(b) != 1:
            continue

        side_effect_records.append({
            "cv": cv_name,
            "model": model_name,
            "PR_sensors_only": float(
                a["PR_AUC"].iloc[0]
            ),
            "PR_sensors_plus_side": float(
                b["PR_AUC"].iloc[0]
            ),
            "PR_delta_side": float(
                b["PR_AUC"].iloc[0]
                - a["PR_AUC"].iloc[0]
            ),
            "ROC_sensors_only": float(
                a["ROC_AUC"].iloc[0]
            ),
            "ROC_sensors_plus_side": float(
                b["ROC_AUC"].iloc[0]
            ),
            "ROC_delta_side": float(
                b["ROC_AUC"].iloc[0]
                - a["ROC_AUC"].iloc[0]
            ),
        })

side_effect_df = pd.DataFrame(
    side_effect_records
)

if len(side_effect_df):
    print(
        side_effect_df
        .round(4)
        .to_string(index=False)
    )

    side_effect_df.to_csv(
        OUT_DIR / "side_effect_summary.csv",
        index=False,
        encoding="utf-8-sig",
    )

# =============================================================================
# 11. 자동 결론
# =============================================================================

banner("10) 자동 결론")

lines = []

lines.append(
    f"- 데이터: {len(df)}행, 불량 {int(y.sum())}개 "
    f"({y.mean():.3%})"
)

lines.append(
    f"- 비상수 센서: {len(sensor_cols)}개 "
    f"(제거: {constant_cols})"
)

lines.append(
    f"- 동일 sensor vector group: "
    f"{group_stats['sensor_group_id'].nunique()}개"
)

lines.append(
    f"- positive 포함 group: "
    f"{int((group_stats['defects'] >= 1).sum())}개"
)

lines.append(
    f"- same-X/different-target group: "
    f"{int((group_stats['target_nunique'] > 1).sum())}개"
)

for _, r in summary_df.iterrows():
    lines.append(
        f"- {r['cv']} | {r['feature_set']} | {r['model']} "
        f"=> PR-AUC={r['PR_AUC']:.4f}, "
        f"ROC-AUC={r['ROC_AUC']:.4f}"
    )

if len(side_effect_df):
    for _, r in side_effect_df.iterrows():
        lines.append(
            f"- Side 효과 [{r['cv']} / {r['model']}]: "
            f"PR-AUC Δ={r['PR_delta_side']:+.4f}"
        )

strict_possible = (
    forward_df["strict_forward_valid"].any()
    if len(forward_df)
    else False
)

lines.append(
    f"- Strict forward 평가 가능 여부: "
    f"{strict_possible}"
)

if not strict_possible:
    lines.append(
        "- 이유: 이후 날짜 validation에 불량 클래스가 없거나, "
        "과거 train에 두 클래스가 모두 존재하지 않음."
    )

for line in lines:
    print(line)

with open(
    OUT_DIR / "validation_summary.txt",
    "w",
    encoding="utf-8",
) as f:
    f.write("\n".join(lines))

# =============================================================================
# 완료
# =============================================================================

banner("완료")

print("결과 폴더:", OUT_DIR)
print()
print("핵심 파일:")
print("  1) validation_summary.csv")
print("  2) side_effect_summary.csv")
print("  3) repeated_group_fold_metrics.csv")
print("  4) repeated_group_oof.csv")
print("  5) purged_block_fold_metrics.csv")
print("  6) purged_block_oof.csv")
print("  7) time_block_stats.csv")
print("  8) strict_forward_diagnostic.csv")
print("  9) sensor_group_stats.csv")
print(" 10) validation_summary.txt")
print()
print(
    "다음 단계에서는 validation_summary를 보고 "
    "Baseline 모델을 확정한 뒤 feature engineering으로 넘어갑니다."
)
